In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import pathlib
import dotenv
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from copy import deepcopy
import cartopy.crs as crs
from highres_ta import estimators as models
from highres_ta.preprocessing import load_data, load_config, preprocess_data
from highres_ta.evaluation import scoring, plot_residuals_map, plot_residuals_y, scatter_residuals_vs_feature

ROOT = pathlib.Path(dotenv.find_dotenv("pyproject.toml")).parent


# Reset to Matplotlib defaults
matplotlib.rcdefaults()

# Explicitly use the default style
plt.style.use('default')

trained_model =   models.BaggingCatBoostResidualRegressor.load(
        ROOT / f"models/{'bagged_catboost_residual_modelstructural_optuna_4.pkl'}"
    )

In [ ]:
swapped_salinity_name = "salt_soda"

config = load_config(ROOT / "scripts/example_config.yaml")
config.xname_features = trained_model.feature_names_in_.tolist()
print(trained_model.feature_names_in_.tolist())

raw_df = load_data()

keys = ["expocode", "time", "lat", "lon", "depth"]

valid_original = raw_df[
    raw_df["salinity"].notna()
]

valid_swapped = raw_df[
    raw_df[swapped_salinity_name].notna()
]

common = (
    valid_original[keys]
    .merge(valid_swapped[keys], on=keys)
    .drop_duplicates()
)

raw_common = raw_df.merge(common, on=keys)

keys = ["expocode", "time", "lat", "lon", "depth"]

raw_common= (
    raw_common
    .groupby(keys, as_index=False)
    .mean(numeric_only=True)
)


# 1. Run preprocessing on both (which assigns the multi-index)
original_df = preprocess_data(raw_common.copy(), config)

swapped_config = deepcopy(config)
swapped_config.salinity_name = swapped_salinity_name
swapped_df = preprocess_data(raw_common.copy(), swapped_config)


# 2. Drop the volatile index levels so both DFs share the exact same 4-level index 
levels_to_drop = ["salinity_bin", "is_coastal"] 
original_core = original_df.reset_index(level=levels_to_drop) 
swapped_core = swapped_df.reset_index(level=levels_to_drop) 

# 3. Find the exact matching rows using a basic index intersection 
common_idx = original_core.index.intersection(swapped_core.index) 

# print(len(original_core)) 
# print(len(swapped_core)) 
# print(len(common_idx)) 

# 4. Use .loc to slice both dataframes to the exact same rows in the exact same order
original_df = original_core.loc[common_idx]
swapped_df = swapped_core.loc[common_idx]

# print(len(original_df))
# print(len(swapped_df))


# 4. Generate predictions on the aligned datasets
# (Now both inputs are guaranteed to have identical lengths)
original_pred = trained_model.predict(original_df[config.xname_features])
swapped_pred  = trained_model.predict(swapped_df[config.xname_features])


# 5. Safe delta calculations (shapes will match perfectly now)
original_residuals = original_pred - original_df['talk']
swapped_residuals  = swapped_pred - swapped_df['talk']

delta_sal = swapped_df['salinity'] - original_df['salinity']
delta_pred = swapped_pred - original_pred
delta_residuals = swapped_residuals - original_residuals


print(original_pred.size)
print(swapped_pred.size)
print(original_residuals.size)
print(swapped_residuals.size)
print(delta_pred.size)
print(delta_residuals.size)
print(delta_sal.size)



In [ ]:
from highres_ta.inference import load_raw_inf_data



In [ ]:
swapped_scores = scoring(y_true = swapped_df['talk'], y_pred = swapped_pred).rename(f"{swapped_config.salinity_name}")
original_scores = scoring(y_true = original_df['talk'], y_pred = original_pred).rename("GLODAP")
scores = pd.concat([original_scores, swapped_scores], axis=1)

fig0, ax0 = plt.subplots(figsize=(8, 3))
ax0.axis("off")
ax0.table(scores.round(3), loc="center", cellLoc="center")
ax0.set_title("Model Performance Metrics", fontsize=14, fontweight="bold")
plt.show()

# Salinity and delta salinity distributions

In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (12,10))

# --- Left: salinity distributions ---
ax[0].hist(
    original_df["salinity"],
    bins=50,
    density=True,
    alpha=0.4,
    label=f"GLODAP",
    color = "blue"
)

#original_df["salinity"].plot(kind="kde", ax=ax[0], linewidth=1, label=f"GLODAP sal KDE", color= 'blue')

ax[0].hist(
    swapped_df["salinity"],
    bins=50,
    density=True,
    alpha=0.4,
    label=f"{swapped_config.salinity_name}",
    color = "red"
)

#swapped_df["salinity"].plot(kind="kde", ax=ax[0], linewidth=1, label=f"CCI + SSS KDE", color= 'red')

ax[0].set_title("Salinity Distribution")
ax[0].set_xlabel("Salinity")
ax[0].grid(True)
ax[0].legend()



# --- Right: noise distribution ---

ax[1].hist(
    delta_sal,
    bins=50,
    density=True,
    alpha=0.6
)

mean_delta_sal = np.mean(delta_sal)
med_delta_sal = np.median(delta_sal)

ax[1].axvline(mean_delta_sal, linewidth=2, label=f"mean={mean_delta_sal:.2f}")
ax[1].axvline(med_delta_sal, linestyle="--", linewidth=2, label=f"median={med_delta_sal:.2f}")

ax[1].set_title("Mismatch distribution (∆S = SSS - sal)")
ax[1].set_xlabel("∆S")
ax[1].grid(True)
ax[1].legend()

fig.tight_layout()

# Residuals and delta residuals distribution

In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (12,10))

# --- Left: salinity distributions ---
ax[0].hist(
    original_residuals,
    bins=50,
    density=True,
    alpha=0.4,
    label=f"GLODAP",
    color = "blue"
)

#original_residuals.plot(kind="kde", ax=ax[0], linewidth=1, label=f"GLODAP KDE", color= blue, linestyle=linestyle)

ax[0].hist(
    swapped_residuals,
    bins=50,
    density=True,
    alpha=0.4,
    label=f"{swapped_config.salinity_name}",
    color = "red"
)

#swapped_residuals.plot(kind="kde", ax=ax[0], linewidth=1, label=f"CCI SSS KDE", color= red, linestyle=linestyle)

ax[0].set_title("Residuals (pred-true) Distribution")
ax[0].set_xlabel("Residuals")
ax[0].grid(True)
ax[0].legend()



ax[1].hist(
    delta_residuals,
    bins=50,
    density=True,
    alpha=0.6
)

mean_delta_res = np.mean(delta_residuals)
med_delta_res = np.median(delta_residuals)

ax[1].axvline(mean_delta_res, linewidth=2, label=f"mean={mean_delta_res:.2f}")
ax[1].axvline(med_delta_res, linestyle="--", linewidth=2, label=f"median={med_delta_res:.2f}")

ax[1].set_title("Mismatch distribution (∆residuals = $r_{SSS}$ - $r_{GLODAP}$)")
ax[1].set_xlabel("∆residuals")
ax[1].grid(True)
ax[1].legend()

fig.tight_layout()

In [ ]:
#TODO: make predictions and residuals arrays have coordinates for plotting (maybe look into ready available fcts)
#TODO: 3x3 maps: sal, delta_sal, pred, delta_pred,  residuals, delta_residuals

In [ ]:
lat = swapped_df.index.get_level_values('lat')
lon = swapped_df.index.get_level_values('lon')


fig, axes = plt.subplots(
    nrows=3, 
    ncols=1, 
    figsize=(12, 20), 
    subplot_kw={'projection': crs.PlateCarree()}  # This sends the projection to the axes
)



sal_kwargs = dict(vmin=20, vmax = 40)
plot_residuals_map(original_df['salinity'], lat = lat , lon = lon, ax = axes[0],**sal_kwargs, title = 'GLODAP')
plot_residuals_map(swapped_df['salinity'], lat = lat , lon = lon, ax = axes[1], **sal_kwargs, title = f"{swapped_config.salinity_name}")
plot_residuals_map(delta_sal, lat = lat , lon = lon, ax = axes[2],  **dict(vmin = -5, vmax = 5), title = "∆sal")


plt.show()

#plot_residuals_map(swapped_residuals, lat = lat , lon = lon, ax = )

In [ ]:
fig, axes = plt.subplots(
    nrows=3, 
    ncols=1, 
    figsize=(12, 20), 
    subplot_kw={'projection': crs.PlateCarree()}  # This sends the projection to the axes
)



pred_kwargs = dict(vmin = 2200, vmax = 2500)
plot_residuals_map(original_pred, lat = lat , lon = lon, ax = axes[0], **pred_kwargs, title = 'GLODAP predictions')
plot_residuals_map(swapped_pred, lat = lat , lon = lon, ax = axes[1], **pred_kwargs, title = f"{swapped_config.salinity_name} predictions")
plot_residuals_map(delta_pred, lat = lat , lon = lon, ax = axes[2], **dict(vmin = -40, vmax = 40),  title = "∆ pred")




In [ ]:
fig, axes = plt.subplots(
    nrows=3, 
    ncols=1, 
    figsize=(12, 20), 
    subplot_kw={'projection': crs.PlateCarree()}  # This sends the projection to the axes
)



res_kwargs = dict(vmin = -200, vmax = 200)
plot_residuals_map(original_residuals, lat = lat , lon = lon, ax = axes[0], **res_kwargs, title = 'GLODAP residuals')
plot_residuals_map(swapped_residuals, lat = lat , lon = lon, ax = axes[1],**res_kwargs, title = f"{swapped_config.salinity_name} residuals")
plot_residuals_map(delta_residuals, lat = lat , lon = lon, ax = axes[2],**res_kwargs,  title = "∆residuals")

In [ ]:
fig, axes = plt.subplots(
    nrows=3, 
    ncols=1, 
    figsize=(12, 20), 
    subplot_kw={'projection': crs.PlateCarree()}  # This sends the projection to the axes
)

plot_residuals_map(delta_sal, lat = lat , lon = lon, ax = axes[0],  **dict(vmin = -5, vmax = 5), title = "∆sal")
plot_residuals_map(delta_pred, lat = lat , lon = lon, ax = axes[1], **dict(vmin = -40, vmax = 40),  title = "∆ pred")
plot_residuals_map(delta_residuals, lat = lat , lon = lon, ax = axes[2],**res_kwargs,  title = "∆residuals")

In [ ]:
fig2, axs2 = plt.subplots(
    2, 1, figsize=(12, 7), sharey=True, sharex=True, constrained_layout=True
)

plot_residuals_y(original_df['talk'], original_pred, ax=axs2[0])
axs2[0].set_title("GLODAP salinity Residuals (Predicted - True)", fontsize=14, fontweight="bold")
axs2[0].set_xlabel("Observed Total Alkalinity (µmol/kg)")
axs2[0].set_ylabel("Residual (µmol/kg)")

plot_residuals_y(swapped_df['talk'], swapped_pred, ax=axs2[1])
axs2[1].set_title(f"{swapped_config.salinity_name} Residuals (Predicted - True)", fontsize=14, fontweight="bold")
axs2[1].set_xlabel("Observed Total Alkalinity (µmol/kg)")
axs2[1].set_ylabel("Residual (µmol/kg)")
axs2[1].set_ylabel("")

In [ ]:
fig3, axs3 = plt.subplots(
    2, 1, figsize=(12, 7), sharey=True, sharex=True, constrained_layout=True
)

scatter_residuals_vs_feature(residuals = original_residuals, feature = original_df['salinity'], ax=axs3[0], **dict(color='blue'))
axs3[0].set_title("GLODAP salinity Residuals (Predicted - True)", fontsize=14, fontweight="bold")
axs3[0].set_xlabel("GLODAP salinity")
axs3[0].set_ylabel("Residual (µmol/kg)")

scatter_residuals_vs_feature(residuals = swapped_residuals, feature = swapped_df['salinity'], ax=axs3[1], **dict(color='red'))
axs3[1].set_title(f"{swapped_config.salinity_name} Residuals (Predicted - True)", fontsize=14, fontweight="bold")
axs3[1].set_xlabel(f"{swapped_config.salinity_name}")
axs3[1].set_ylabel("Residual (µmol/kg)")
axs3[1].set_ylabel("")

In [ ]:
import pandas as pd
import pathlib
import dotenv
ROOT = pathlib.Path(dotenv.find_dotenv("pyproject.toml")).parent

all_scores =pd.read_csv(ROOT/"outputs/salinity_swaps/ score_history.csv")
all_scores


swapped_columns = [y for y in all_scores.columns if "swapped" in y]
original_columns = [y for y in all_scores.columns if "original" in y]

metrics_columns_zip = zip(swapped_columns, original_columns)
metrics_columns_zip = list(metrics_columns_zip)

metric_list = []


delta_scores = all_scores[["evaluation_set","n_samples", "salinity_product"]].copy()
relative_delta_scores = all_scores[["evaluation_set","n_samples", "salinity_product"]].copy()

for swapped_col, original_col in metrics_columns_zip:
    
    metric = swapped_col.split("_")[1]
    if "bias" in metric:
        metric = metric.split(" ")[0] + " bias"
    else:
        metric = metric.split(" ")[0]  # Extract the metric name (e.g., "rmse")
    
    metric_list.append(metric)
    
    delta_scores[rf"$\Delta$ {metric}"] = all_scores[swapped_col] - all_scores[original_col]
    delta_scores[rf"$\Delta$ {metric} (\%)"] = 100*(all_scores[swapped_col] - all_scores[original_col]) / all_scores[original_col]


In [ ]:
delta_scores.columns

In [ ]:
def publish_latex_table(scores, columns_to_publish, output_path):
    
    table_to_publish = scores[columns_to_publish].copy()

    table_to_publish = table_to_publish.rename(columns={
        "salinity_product": "Salinity product",
        "n_samples": "Count",
    })

        # format all numeric columns explicitly
    for col in table_to_publish.columns:
        if col != "Salinity product" and col != "Count":
            table_to_publish[col] = table_to_publish[col].map(lambda x: f"{x:.2f}")


    from pathlib import Path

    if 'test' in output_path.name:
        caption = "Relative performance changes between original and swapped salinity products. Metrics are computed on the (GLODAP Test set $\cap$ swapped product) subset."
        label = "tab:salinity_swap_metrics_test"
        label = output_path.name.replace(".tex", "")
    else:
        caption = "Relative performance changes between original and swapped salinity products. Metrics are computed on the (GLODAP $\cap$ swapped product) subset."
        label = output_path.name.replace(".tex", "")
        
    latex_table = table_to_publish.to_latex(
        index=False,
        escape=False,          # IMPORTANT for LaTeX math ($\Delta$, $R^2$)
        column_format="lcccccc",
        caption= caption,
        label=label,
        bold_rows=False,
    )

    # Optional: improve readability with booktabs (recommended)
    latex_table = latex_table.replace("\\toprule", "\\toprule\n\\midrule", 1)

    output_path = output_path if isinstance(output_path, Path) else Path(output_path)
    output_path.write_text(latex_table)

    print(latex_table)

In [ ]:
groups = delta_scores.groupby("evaluation_set")
delta_test_scores = groups.get_group("Test set")
delta_whole_scores = groups.get_group("Whole dataset")




columns_to_publish = ["salinity_product", "n_samples", rf"$\Delta$ MAE (\%)", rf"$\Delta$ RMSE (\%)", rf"$\Delta$ Mean bias", rf"$\Delta$ $R^2$"] 

publish_latex_table(delta_test_scores, columns_to_publish, output_path = ROOT / "outputs/salinity_swaps/test_set/comparison_test_metrics.tex")
publish_latex_table(delta_whole_scores, columns_to_publish, output_path = ROOT / "outputs/salinity_swaps/whole_dataset/comparison_wholeset_metrics.tex")

# scores on common dataset

In [ ]:
metrics_dict = {
        "r2_score": r"$R^2$",
        "mean_absolute_error": r"MAE ($\mu mol\ kg^{-1}$)",
        "root_mean_squared_error": r"RMSE ($\mu mol\ kg^{-1}$)",
        "median_absolute_error": r"MedAE ($\mu mol\ kg^{-1}$)",
        "huber_loss": "Huber",
        "mean_bias": r"Mean bias ($\mu mol\ kg^{-1}$)",
        "median_bias": r"Median bias ($\mu mol\ kg^{-1}$)",
    }



from highres_ta import estimators as models
from highres_ta.utils import ROOT
from highres_ta.evaluation import scoring
import pandas as pd

MODEL_NAME = "bagged_catboost_residual_modelstructural_optuna_4"

OUTPUT_DIR = (
        ROOT
        / f"outputs/salinity_swaps/common_intersection"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_dir = ROOT/"data/bagged_catboost_residual_modelstructural_optuna_4/salinity_swap/common_intersection"

product_dict = {
        "GLODAP": "GLODAP",
        "sss_cci": "SSS CCI",
        "salt_soda": "SODA",
        "sss_glorys": "SSS GLORYS",
        "sss_multiobs": "SSS MultiObs",
    }

trained_model = (
    models.BaggingCatBoostResidualRegressor.load(
        ROOT/ f"models/{MODEL_NAME}.pkl"
    )
)

xname_features = trained_model.feature_names_in_.tolist()


original_common_df = pd.read_parquet(data_dir / "original_df.pq")

original_common_pred = trained_model.predict(original_common_df[xname_features])


original_scores = scoring(
        y_true=original_common_df["talk"],
        y_pred=original_common_pred,
    ).rename(f"GLODAP").rename(index=metrics_dict)

salinity_products = [
    "sss_cci",
    "salt_soda",
    "sss_glorys",
    "sss_multiobs",
]

swapped_common_preds = {}
swapped_common_dfs = {}
swapped_scores = {}
swapped_common_residuals= {}

for product in salinity_products:
    
    swapped_common_df = pd.read_parquet(data_dir / f"{product}_swapped_df.pq")
    swapped_common_dfs[product] = swapped_common_df
    swapped_common_preds[product] = trained_model.predict(swapped_common_df[xname_features])
    swapped_common_residuals[product] = swapped_common_preds[product] - swapped_common_df['talk']
    swapped_scores[product] = scoring(
        y_true=swapped_common_df["talk"],
        y_pred=swapped_common_preds[product],
    ).rename(f"{product}").rename(index=metrics_dict)

all_scores = pd.concat(
    [original_scores] +
    [swapped_scores[product] for product in salinity_products],
    axis=1,
).rename(
    columns= product_dict
)



all_scores.to_csv(OUTPUT_DIR / "all_scores.csv")

latex_table = all_scores.to_latex(
    float_format="%.2f",
    escape=False,
    column_format="l" + "c" * len(all_scores.columns),
    caption=f"Performance metrics for the GLODAP and salinity-product swaps, on the intersection of all datasets (N={len(original_common_pred)} samples).",
    label="tab:salinity_swap_scores_common_intersection",
)

with open(OUTPUT_DIR / "salinity_swap_scores.tex", "w") as f:
    f.write(latex_table)


In [ ]:
from highres_ta.evaluation import plot_residuals_map
from matplotlib import pyplot as plt
import matplotlib
import cmocean.cm as cmo

# Reset to Matplotlib defaults
matplotlib.rcdefaults()

# Explicitly use the default style
plt.style.use('default')

pred_kwargs = dict(vmin=2200, vmax=2500, cmap=cmo.thermal)

plot_residuals_map(
    original_common_pred,
    lat=original_common_df.index.get_level_values("lat"),
    lon=original_common_df.index.get_level_values("lon"),
    ax=None,
    title="GLODAP predictions - common intersection",
    **pred_kwargs
)



In [ ]:
from highres_ta.evaluation import plot_residuals_map
from matplotlib import pyplot as plt
import matplotlib
import cmocean.cm as cmo

# Reset to Matplotlib defaults
matplotlib.rcdefaults()

# Explicitly use the default style
plt.style.use('default')


def save_figure(fig, name):

    fig.savefig(
        OUTPUT_DIR / f"{name}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

res_kwargs = dict(vmin = -40, vmax = 40, cmap=cmo.balance)


plot_residuals_map(
        original_common_pred - original_common_df['talk'],
        lat=original_common_df.index.get_level_values("lat"),
        lon=original_common_df.index.get_level_values("lon"),
        ax=None,
        title=f"GLODAP residuals (predicted - true) - common intersection",
        **res_kwargs
    )
    
plt.show();


for product in salinity_products:
    
    fig1, ax1 = plt.subplots(
        figsize=(10, 5.5),
        constrained_layout=True,
        subplot_kw={"projection": crs.PlateCarree()},
    )
    
    plot_residuals_map(
        swapped_common_residuals[product],
        lat=swapped_common_dfs[product].index.get_level_values("lat"),
        lon=swapped_common_dfs[product].index.get_level_values("lon"),
        ax=ax1,
        title=f"{product} residuals (predicted - true) - common intersection",
        **res_kwargs
    )
    
    ax1.coastlines(linewidth=0.6)

    gl = ax1.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    
    
    save_figure(fig1, f"residuals_map_{product}_common_intersection")


    
    fig.savefig(OUTPUT_DIR / f"{product}_residuals_map.png", dpi=300)
    
    

In [ ]:
all_scores